In [ ]:
# Install any libraries not pre-installed in Colab
!pip install -q grad-cam
!pip install -q kaggle --upgrade

# Core imports
import os
import json
import zipfile
import subprocess
from pathlib import Path

import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.data import DataLoader, ConcatDataset, random_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Confirm GPU
print(f"✅ GPU enabled: {torch.cuda.is_available()}")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ Torchvision version: {torchvision.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Now your Drive is accessible at:
# /content/drive/MyDrive/

In [ ]:
import subprocess
from pathlib import Path
from google.colab import userdata

git_email = userdata.get("GIT_EMAIL")
git_name = userdata.get("GIT_NAME")
repo_url = userdata.get("REPO_URL")
github_token = userdata.get("GITHUB_TOKEN")

# Configure git identity
!git config --global user.email "{git_email}"
!git config --global user.name "{git_name}"

# Build authenticated URL
repo_url_with_token = repo_url.replace("https://", f"https://{github_token}@")

repo_dir = Path("/content") / Path(repo_url).stem

if not repo_dir.exists():
    !git clone {repo_url_with_token} {repo_dir}
else:
    print("Repo already exists. Pulling latest changes...")
    !git -C {repo_dir} remote set-url origin {repo_url_with_token}
    !git -C {repo_dir} pull

# Reset to clean URL (no token stored)
!git -C {repo_dir} remote set-url origin {repo_url}

os.chdir(repo_dir)
print("Current directory:", os.getcwd())

In [ ]:
base = '/content/drive/MyDrive/chest_xray_project'

for folder in ['data/train', 'data/val', 'data/test', 'models', 'results', 'notebooks']:
    os.makedirs(f'{base}/{folder}', exist_ok=True)

print("Folders created!")

In [ ]:
import numpy as np
import pandas as pd

print("=== PHASE 1 CHECKLIST ===")

# Check GPU
gpu = torch.cuda.is_available()
print(f"✓ GPU enabled: {gpu}")

# Check libraries
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Torchvision version: {torchvision.__version__}")

# Check Drive mounted
drive_mounted = os.path.exists('/content/drive/MyDrive')
print(f"✓ Google Drive mounted: {drive_mounted}")

# Check project folders
folders = ['data/train', 'data/val', 'data/test', 'models', 'results', 'notebooks']
base = '/content/drive/MyDrive/chest_xray_project'
all_good = True
for folder in folders:
    exists = os.path.exists(f'{base}/{folder}')
    print(f"✓ {folder}: {exists}")
    if not exists:
        all_good = False

# Check git
import subprocess
name = subprocess.getoutput('git config --global user.name')
email = subprocess.getoutput('git config --global user.email')
print(f"✓ Git name: {name}")
print(f"✓ Git email: {email}")

print("")
if all_good and drive_mounted and gpu:
    print("All checks passed! Ready for Phase 2.")
else:
    print("Something needs fixing — let me know what printed False.")

In [ ]:
gitignore_content = """# Kaggle credentials
/root/.kaggle/

# Data & outputs — too large for GitHub
*.zip
data/
results/
models/*.pth

# Notebook clutter
.ipynb_checkpoints/
__pycache__/
*.pyc
"""

gitignore_path = repo_dir / ".gitignore"

with open(gitignore_path, "w") as f:
    f.write(gitignore_content)

print("✅ .gitignore written!")

In [ ]:
import json
from pathlib import Path
from google.colab import userdata

# Store these in Colab Secrets, not in the notebook:
# KAGGLE_USERNAME = your Kaggle username
# KAGGLE_KEY = your Kaggle API key

kaggle_username = userdata.get("KAGGLE_USERNAME")
kaggle_key = userdata.get("KAGGLE_KEY")

if not kaggle_username or not kaggle_key:
    raise ValueError("Missing KAGGLE_USERNAME or KAGGLE_KEY in Colab Secrets.")

kaggle_dir = Path("/root/.kaggle")
kaggle_dir.mkdir(exist_ok=True)

with open(kaggle_dir / "kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)

os.chmod(kaggle_dir / "kaggle.json", 0o600)

print("Kaggle configured!")

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/drive/MyDrive/chest_xray_project/data
print("Download complete!")

In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/chest_xray_project/data/chest-xray-pneumonia.zip'
extract_path = '/content/drive/MyDrive/chest_xray_project/data/'
extract_check = extract_path + 'chest_xray/train'

if not os.path.exists(extract_check):
    print("Unzipping...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Done!")
else:
    print("✅ Already extracted, skipping.")

In [ ]:
base = '/content/drive/MyDrive/chest_xray_project/data/chest_xray'

for split in ['train', 'val', 'test']:
    normal = len(os.listdir(f'{base}/{split}/NORMAL'))
    pneumonia = len(os.listdir(f'{base}/{split}/PNEUMONIA'))
    print(f"{split.upper()}")
    print(f"  Normal:    {normal} images")
    print(f"  Pneumonia: {pneumonia} images")
    print(f"  Total:     {normal + pneumonia} images")
    print()

In [ ]:
base = '/content/drive/MyDrive/chest_xray_project/data/chest_xray'

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Chest X-Rays', fontsize=16)

for i, category in enumerate(['NORMAL', 'PNEUMONIA']):
    folder = f'{base}/train/{category}'
    images = os.listdir(folder)[:4]
    for j, img_name in enumerate(images):
        img = Image.open(f'{folder}/{img_name}').convert('RGB')
        axes[i][j].imshow(img, cmap='gray')
        axes[i][j].set_title(category)
        axes[i][j].axis('off')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/chest_xray_project/results/sample_xrays.png')
plt.show()
print("Saved to results folder!")

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset, random_split

base = '/content/drive/MyDrive/chest_xray_project/data/chest_xray'

# Define transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Transforms defined!")

In [ ]:
# ── Data Loaders ──────────────────────────────────────────────────────────────
from torch.utils.data import WeightedRandomSampler

base = '/content/drive/MyDrive/chest_xray_project/data/chest_xray'

# Merge train + val, then re-split 80/20
train_dataset_raw = datasets.ImageFolder(f'{base}/train', transform=train_transforms)
val_dataset_raw   = datasets.ImageFolder(f'{base}/val',   transform=train_transforms)
combined          = ConcatDataset([train_dataset_raw, val_dataset_raw])

total_len = len(combined)
val_len   = int(0.2 * total_len)
train_len = total_len - val_len
train_dataset, val_dataset = random_split(combined, [train_len, val_len],
                                           generator=torch.Generator().manual_seed(42))

test_dataset = datasets.ImageFolder(f'{base}/test', transform=test_transforms)

# ── Dynamic class weights ─────────────────────────────────────────────────────
combined_targets        = train_dataset_raw.targets + val_dataset_raw.targets
combined_targets_tensor = torch.tensor(combined_targets)
class_counts            = torch.bincount(combined_targets_tensor)
class_weights           = [1.0 / c.item() for c in class_counts]

print(f"Normal count:    {class_counts[0].item()}")
print(f"Pneumonia count: {class_counts[1].item()}")
print(f"Class weights:   {class_weights}")

# ── Weighted sampler ──────────────────────────────────────────────────────────
train_indices  = train_dataset.indices
sample_weights = [class_weights[combined_targets[i]] for i in train_indices]
sampler        = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")
print("Data loaders ready!")

In [ ]:
# Count classes in training set
normal = 1341
pneumonia = 3875
total = normal + pneumonia

print(f"Normal:    {normal} ({100*normal/total:.1f}%)")
print(f"Pneumonia: {pneumonia} ({100*pneumonia/total:.1f}%)")

# Plot
plt.figure(figsize=(6,4))
plt.bar(['Normal', 'Pneumonia'], [normal, pneumonia], color=['steelblue', 'coral'])
plt.title('Class Distribution in Training Set')
plt.ylabel('Number of Images')
plt.savefig('/content/drive/MyDrive/chest_xray_project/results/class_distribution.png')
plt.show()
print("Saved to results folder!")

In [ ]:
import torch.nn as nn
from torchvision import models

# Load pre-trained ResNet50
model = models.resnet50(weights='IMAGENET1K_V1')

# Freeze all layers except the final one
for param in model.parameters():
    param.requires_grad = False

# Replace final layer for binary classification
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 2)
)

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model loaded on: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("\nModel ready!")

In [ ]:
import torch.optim as optim

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer - only update the final layer we replaced
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Learning rate scheduler - reduces LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=3,
    factor=0.1
)

print("Training configuration:")
print(f"  Loss function: Cross Entropy")
print(f"  Optimizer:     Adam")
print(f"  Learning rate: 0.001")
print(f"  Scheduler:     ReduceLROnPlateau")
print("\nReady to train!")

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = float('inf')
        self.counter    = 0
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            print(f"  EarlyStopping: no improvement for {self.counter}/{self.patience} epochs")
            if self.counter >= self.patience:
                self.should_stop = True

In [ ]:
MODEL_PATH    = '/content/drive/MyDrive/chest_xray_project/models/best_model.pth'
FORCE_RETRAIN = False  # Set True to retrain even if a saved model exists

if os.path.exists(MODEL_PATH) and not FORCE_RETRAIN:
    mtime = os.path.getmtime(MODEL_PATH)
    saved_time = pd.Timestamp(mtime, unit='s').strftime('%Y-%m-%d %H:%M:%S')
    print(f"⚠️  Loading saved model from {saved_time}")
    print("    Set FORCE_RETRAIN = True if you've changed the architecture or transforms.")

if os.path.exists(MODEL_PATH) and not FORCE_RETRAIN:
    print("✅ Saved model found. Loading instead of retraining...")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device)['model_state_dict'])
    model.eval()
    print("✅ Model loaded and ready!")

else:
    print("🏋️ No saved model found, training from scratch...")
    num_epochs = 10
    best_val_loss = float('inf')
    train_losses, val_losses, val_accuracies = [], [], []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss = val_loss / len(val_loader)
        val_acc = 100 * correct / total
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        # Update scheduler
        scheduler.step(val_loss)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
    'model_state_dict': model.state_dict(),
    'class_to_idx':     train_dataset_raw.class_to_idx,
}, MODEL_PATH)
            saved = "✓ saved"
        else:
            saved = ""

        print(f"Epoch {epoch+1:2d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.1f}% {saved}")

    print("\nTraining complete!")
    print(f"Best validation loss: {best_val_loss:.4f}")

In [ ]:
from sklearn.metrics import (roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
import numpy as np

# Load best model
checkpoint = torch.load(
    '/content/drive/MyDrive/chest_xray_project/models/best_model.pth',
    map_location=device, weights_only=False)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("✅ Model loaded!")

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())
        all_probs.extend(probs[:,1].cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

# Print results
auc = roc_auc_score(all_labels, all_probs)
print("=== TEST SET RESULTS ===")
print(f"AUC-ROC:  {auc:.4f}")
print()
print(classification_report(all_labels, all_preds,
                            target_names=['Normal', 'Pneumonia']))

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='coral', lw=2,
         label=f'ROC curve (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve — Pneumonia Classifier')
plt.legend(loc='lower right')
plt.savefig('/content/drive/MyDrive/chest_xray_project/results/roc_curve.png')
plt.show()
print("Saved to results folder!")

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Pneumonia'],
            yticklabels=['Normal', 'Pneumonia'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Pneumonia Classifier')
plt.savefig('/content/drive/MyDrive/chest_xray_project/results/confusion_matrix.png')
plt.show()
print("Saved to results folder!")

In [ ]:
import torch.nn as nn
from torchvision import models, transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import numpy as np

# Reload model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.resnet50(weights='IMAGENET1K_V1')
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 2)
)
model.load_state_dict(torch.load(
    '/content/drive/MyDrive/chest_xray_project/models/best_model.pth',
    map_location=device, weights_only=False)['model_state_dict'])
model = model.to(device)
model.eval()
print("Model reloaded!")

# Set up Grad-CAM
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

# Load sample image
sample_path = '/content/drive/MyDrive/chest_xray_project/data/chest_xray/test/PNEUMONIA'
sample_img_path = f'{sample_path}/{os.listdir(sample_path)[0]}'
img = Image.open(sample_img_path).convert('RGB')
img_resized = img.resize((224, 224))
img_array = np.array(img_resized) / 255.0

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
input_tensor = transform(img).unsqueeze(0).to(device)

# Get prediction
with torch.no_grad():
    output = model(input_tensor)
    pred_idx = output.argmax(dim=1).item()
    confidence = torch.softmax(output, dim=1)[0, pred_idx].item()

CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
print(f"Predicted: {CLASS_NAMES[pred_idx]} ({confidence:.1%})")

# Generate heatmap
targets = [ClassifierOutputTarget(pred_idx)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
grayscale_cam = grayscale_cam[0, :]
visualization = show_cam_on_image(img_array.astype(np.float32),
                                   grayscale_cam, use_rgb=True)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_resized, cmap='gray')
axes[0].set_title('Original X-Ray')
axes[0].axis('off')
axes[1].imshow(visualization)
axes[1].set_title('Grad-CAM Heatmap')
axes[1].axis('off')
plt.suptitle('Where the model looks to detect pneumonia', fontsize=14)
plt.savefig('/content/drive/MyDrive/chest_xray_project/results/gradcam.png')
plt.show()
print("Saved to results folder!")

In [ ]:
from google.colab import userdata

github_token = userdata.get("GITHUB_TOKEN")
repo_url = userdata.get("REPO_URL")

repo_url_with_token = repo_url.replace("https://", f"https://{github_token}@")

!git remote set-url origin {repo_url_with_token}
!git pull origin main --rebase
!git push origin main

# Clean up token from remote URL
!git remote set-url origin {repo_url}
print("✅ Done!")